In [ ]:
import pandas as pd

: 

In [ ]:
df4_with_regions = pd.read_pickle('../data/df4_with_regions.pkl')

: 

In [ ]:
df5 = pd.read_pickle('../data/df5.pkl')

final termination reasons by patron category

In [ ]:
df4_with_regions.groupby('PATRON_CATG_DESC_TXT')['final_termination_reason_spatial'].value_counts().reset_index(name='count')

final termination reasons by patron category (percentage)

In [ ]:
# percentage by patron category
df4_with_regions.groupby('PATRON_CATG_DESC_TXT')['final_termination_reason_spatial'] \
    .value_counts(normalise=True) \
    .reset_index(name='pct_within_group')

In [ ]:
# overall percentage
df4_with_regions['final_termination_reason_spatial'] \
    .value_counts(normalize=True) \
    .reset_index(name='pct_overall') \
    .rename(columns={'index': 'final_termination_reason_spatial'})

overall transfer metrics by patron category

In [ ]:
def get_df_val(data, correct_data):
    df = data[['CRD_NUM', 'JRNY_ID_NUM', 'ENTRY_TM', 'EXIT_TM', 'TRNSPT_MODE_CD', 'journey_termination_flag', 
    'ORIG_STATION_NAME', 'dest_region']]
    correct_df = correct_data[['CRD_NUM', 'JRNY_ID_NUM', 'JRNY_START_TM', 'JRNY_END_TM', 'PATRON_CATG_ID_NUM']]

    correct_df['service_day'] = (correct_df['JRNY_START_TM'] - pd.Timedelta(hours=5)).dt.date
    target_day = pd.Timestamp('2025-02-12').date()
    correct_df = correct_df[correct_df['service_day'] == target_day].reset_index(drop=True)
    correct_df = correct_df[correct_df['PATRON_CATG_ID_NUM'].isin([1, 3, 4])].reset_index(drop=True)

    df_val = df.merge(
        correct_df[['CRD_NUM', 'JRNY_ID_NUM', 'JRNY_START_TM', 'JRNY_END_TM']],
        on=['CRD_NUM', 'JRNY_ID_NUM'],
        how='inner'
    )

    df_val = df_val[
        (df_val['ENTRY_TM'] >= df_val['JRNY_START_TM']) &
        (df_val['EXIT_TM'] <= df_val['JRNY_END_TM'])
    ].reset_index(drop=True)

    df_val = df_val.sort_values(['CRD_NUM', 'ENTRY_TM']).reset_index(drop=True)
    df_val['next_JRNY_ID_NUM'] = df_val.groupby('CRD_NUM')['JRNY_ID_NUM'].shift(-1)
    df_val['true_transfer'] = (df_val['JRNY_ID_NUM'] == df_val['next_JRNY_ID_NUM'])
    df_val = df_val[df_val['next_JRNY_ID_NUM'].notna()].reset_index(drop=True)

    return df_val    

In [ ]:
def get_metrics(df):
    actual_transfer = df['true_transfer']
    pred_transfer = ~df['journey_termination_flag_spatial']

    tp = (actual_transfer & pred_transfer).sum()     # actual transfer, predicted transfer
    tn = (~actual_transfer & ~pred_transfer).sum()   # actual new journey, predicted new journey
    fp = (~actual_transfer & pred_transfer).sum()    # actual new journey, predicted transfer (merge error)
    fn = (actual_transfer & ~pred_transfer).sum()    # actual transfer, predicted new journey (split error)

    total_actual_transfers = actual_transfer.sum()

    print(f"TP: {tp:,}  TN: {tn:,}  FP: {fp:,}  FN: {fn:,}")
    print(f"Split rate  (FN / actual transfers): {fn / total_actual_transfers:.4f}")
    print(f"Merge rate  (FP / actual transfers): {fp / total_actual_transfers:.4f}")
    print(f"Sensitivity (TP / (TP+FN)):          {tp / (tp + fn):.4f}")
    print(f"Specificity (TN / (TN+FP)):          {tn / (tn + fp):.4f}")
    print(f"Accuracy    ((TP+TN) / total):       {(tp + tn) / len(df):.4f}")

    print(pd.crosstab(
        actual_transfer,
        pred_transfer,
        rownames=['Actual transfer'],
        colnames=['Predicted transfer']
    ))

In [ ]:
# if needed
%who
del df4_with_regions, df5

In [ ]:
df_val = get_df_val(df4_with_regions, df5)
get_metrics(df_val)

In [ ]:
for group_name, group_df in df_val.groupby('PATRON_CATG_DESC_TXT'):
    print(f"\nGroup: {group_name}")
    get_metrics(group_df)

getting wrongly split and wrongly merged rides from our classifier

In [ ]:
def get_misclassified(df_val):
    actual_transfer = df_val['true_transfer']
    pred_transfer = ~df_val['journey_termination_flag_spatial']    

    fp_rows = (~actual_transfer & pred_transfer)
    fn_rows = (actual_transfer & ~pred_transfer)

    fp_df = df_val[fp_rows]
    fn_df = df_val[fn_rows]

    return fp_df, fn_df

In [ ]:
fp, fn = get_misclassified(df4_with_regions, df5)

final termination reasons by patron category [wrongly merged only]

In [ ]:
fp.groupby('PATRON_CATG_DESC_TXT')['final_termination_reason_spatial'].value_counts().reset_index(name='count')

final termination reasons by patron category (percentage) [wrongly merged only]

In [ ]:
# percentage by patron category
fp.groupby('PATRON_CATG_DESC_TXT')['final_termination_reason_spatial'] \
    .value_counts(normalise=True) \
    .reset_index(name='pct_within_group')

In [ ]:
# overall percentage
fp['final_termination_reason_spatial'] \
    .value_counts(normalize=True) \
    .reset_index(name='pct_overall') \
    .rename(columns={'index': 'final_termination_reason_spatial'})